# SVM and Naive Bayes

Assignment answers  |  Source: ml7.pdf

All questions are answered in slightly expanded, student-friendly language. Code blocks are runnable examples; dataset-dependent outputs should be generated locally.

## Question 1. What is a Support Vector Machine (SVM), and how does it work?

Answer: A Support Vector Machine finds a separating hyperplane that maximizes the margin between classes. The observations closest to the boundary are support vectors and strongly influence its position. With kernels, an SVM can also learn nonlinear decision boundaries. It is useful for text classification and other high-dimensional problems.

## Question 2. Explain the difference between Hard Margin and Soft Margin SVM.

Answer: Hard-margin SVM requires every training observation to be classified correctly and works only when classes are linearly separable. Soft-margin SVM allows margin violations and misclassifications using a penalty controlled by C. A smaller C allows more violations and stronger regularization, while a larger C emphasizes training fit.

## Question 3. What is the Kernel Trick in SVM? Give one example of a kernel and explain its use case.

Answer: The kernel trick computes similarities as though observations were mapped into a higher-dimensional feature space without explicitly constructing that mapping. An RBF kernel can capture curved decision boundaries, such as circular class regions. Kernel selection and parameter tuning should use cross-validation.

## Question 4. What is a Naïve Bayes Classifier, and why is it called “naïve”?

Answer: Naive Bayes is a probabilistic classifier based on Bayes’ theorem. It is called naive because it assumes predictors are conditionally independent given the class, which is often not literally true. Despite this simplification, it can perform well on high-dimensional text data and is fast to train.

## Question 5. Describe the Gaussian, Multinomial, and Bernoulli Naïve Bayes variants. When would you use each one?

Answer: Gaussian Naive Bayes assumes continuous feature values follow a normal distribution within each class, so it suits many numeric measurements. Multinomial Naive Bayes models count-like features, commonly word counts in documents. Bernoulli Naive Bayes models binary indicators, such as whether a word appears in an email.

## Question 6. Write a Python program to: ●​ Load the Iris dataset ●​ Train an SVM Classifier with a linear kernel ●​ Print the model's accuracy and support vectors.

Answer: Train a linear SVM on Iris. The support vectors are the training samples nearest the separating boundaries, represented in the fitted feature space.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
X,y=load_iris(return_X_y=True)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.3,stratify=y,random_state=42)
model=make_pipeline(StandardScaler(),SVC(kernel='linear'))
model.fit(Xtr,ytr)
print('Accuracy:',model.score(Xte,yte))
print('Support vectors (scaled):\n',model[-1].support_vectors_)

## Question 7. Write a Python program to: ●​ Load the Breast Cancer dataset ●​ Train a Gaussian Naïve Bayes model ●​ Print its classification report including precision, recall, and F1-score.

Answer: Gaussian Naive Bayes estimates a class-conditional normal distribution for each feature. The classification report gives precision, recall and F1 by class.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report
X,y=load_breast_cancer(return_X_y=True)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
model=GaussianNB().fit(Xtr,ytr)
print(classification_report(yte,model.predict(Xte)))

## Question 8. Write a Python program to: ●​ Train an SVM Classifier on the Wine dataset using GridSearchCV to find the best C and gamma. ●​ Print the best hyperparameters and accuracy.

Answer: Use a pipeline so StandardScaler is fitted within each cross-validation fold. Search C and gamma for an RBF-kernel SVM and evaluate the selected model on held-out Wine samples.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
X,y=load_wine(return_X_y=True)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.3,stratify=y,random_state=42)
pipe=make_pipeline(StandardScaler(),SVC(kernel='rbf'))
search=GridSearchCV(pipe,{'svc__C':[.1,1,10],
    'svc__gamma':['scale',.01,.1,1]},cv=5)
search.fit(Xtr,ytr)
print('Best:',search.best_params_)
print('Accuracy:',search.score(Xte,yte))

## Question 9. Write a Python program to: ●​ Train a Naïve Bayes Classifier on a synthetic text dataset (e.g. using sklearn.datasets.fetch_20newsgroups). ●​ Print the model's ROC-AUC score for its predictions.

Answer: Use a small synthetic text dataset with binary spam labels. Multinomial Naive Bayes produces class probabilities for computing ROC-AUC; the small dataset is only a demonstration.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
texts=['win free prize','meeting at noon','claim cash now','project update',
       'limited free offer','schedule review','win cash today','team lunch',
       'urgent prize claim','report attached','free bonus','client meeting']
y=[1,0,1,0,1,0,1,0,1,0,1,0]
tr,te=train_test_split(range(len(texts)),test_size=.33,stratify=y,random_state=42)
vectorizer=CountVectorizer()
Xtr=vectorizer.fit_transform([texts[i] for i in tr])
Xte=vectorizer.transform([texts[i] for i in te])
model=MultinomialNB().fit(Xtr,[y[i] for i in tr])
print('ROC-AUC:',roc_auc_score([y[i] for i in te],model.predict_proba(Xte)[:,1]))

## Question 10. Imagine you’re working as a data scientist for a company that handles email communications.​ Your task is to automatically classify emails as Spam or Not Spam. The emails may contain: ●​ Text with diverse vocabulary ●​ Potential class imbalance (far more legitimate emails than spam) ●​ Some incomplete or missing data Explain the approach you would take to: ●​ Preprocess the data (e.g. text vectorization, handling missing data) ●​ Choose and justify an appropriate model (SVM vs. Naïve Bayes) ●​ Address class imbalance ●​ Evaluate the performance of your solution with suitable metrics​ And explain the business impact of your solution.

Answer: For spam detection, clean empty emails and remove duplicates before a stratified split. Convert text into TF-IDF features; compare Multinomial Naive Bayes (fast baseline) against linear SVM (often strong with sparse text). Handle imbalance using class weights for SVM or training-only resampling, then tune on cross-validation. Measure spam precision, recall, F1 and PR-AUC to balance unwanted spam against legitimate messages being blocked.